## License
Copyright 2026 jphall@gwu.edu. MIT License; see the repository LICENSE file.

# Basic Azure AI Foundry configuration and connection

This notebook provides a small starting point for connecting to a GW Azure AI Foundry resource. It demonstrates the Chat Completions API, the Responses API, and a single embedding request. These are the same connection examples used in the GW Azure AI Foundry Student Sandbox Access Guide.

Use only the resource assigned to your group and set GW_AZURE_OPENAI_KEY in your environment.


## 1. Import packages


In [5]:
# 1. Import packages
# Load the Azure OpenAI client and secure key helper used by all examples.

import os
from getpass import getpass

from openai import AzureOpenAI


## 2. Chat Completion

Replace only RESOURCE with the resource assigned to your group.


In [6]:
# 2. Chat Completion
# Send the guide's short database haiku prompt through the Chat Completions API.

RESOURCE = "gw-sb-01"  # Replace only with your assigned resource.
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=api_key,
    api_version="2024-12-01-preview",
)

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about databases."},
    ],
    # Reserve visible-output space after GPT-5 Nano's internal reasoning.
    max_completion_tokens=1600,
)

answer = response.choices[0].message.content or ""

# Stop clearly rather than silently printing an empty response.
if not answer.strip():
    raise RuntimeError("No visible response was returned. Please run the request again.")

print(answer)


Rows whisper softly
Indexing threads through data
Facts bloom in queries


## 3. Responses API

The minimal reasoning setting and token limit leave space for visible text after GPT-5 internal reasoning.


In [7]:
# 3. Responses API
# Ask the guide's database-index question using the Responses API.

RESOURCE = "gw-sb-01"  # Replace only with your assigned resource.
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=api_key,
    api_version="2025-03-01-preview",
)

response = client.responses.create(
    model="gpt-5-mini",
    input="Explain briefly why database indexes can improve query performance.",
    max_output_tokens=1600,
    reasoning={"effort": "minimal"},
)

if not response.output_text:
    raise RuntimeError("No visible response was returned. Please run the request again.")

print(response.output_text)


Database indexes improve query performance by reducing the amount of data the database engine must examine to satisfy a query. Key reasons:

- Fast lookups: An index is a data structure (commonly a B-tree or hash) that lets the database find rows matching a value or range without scanning every row, cutting I/O and CPU.
- Efficient sorting and range queries: Ordered indexes support ORDER BY and BETWEEN queries without extra sorting.
- Selective access: For highly selective predicates (few matching rows), indexes drastically reduce rows scanned.
- Index-only plans: Some queries can be answered using the index alone (covering/index-only), avoiding access to the full table.
- Join speedups: Indexes on join keys let the optimizer use faster join algorithms (index nested-loop, merge join) instead of expensive scans.

Note: indexes add storage and slow writes (INSERT/UPDATE/DELETE), so they should be added where read performance gains outweigh write and storage costs.


## 4. Simple embedding

An embedding is a numeric representation of text. This example sends one short phrase to text-embedding-3-small, then reports the vector length and a few values instead of printing all 1,536 dimensions.


In [8]:
# 4. Simple embedding
# Create one reusable vector with the same Azure client configured above.

text_to_embed = "AI risk management training"

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=[text_to_embed],
)

embedding = embedding_response.data[0].embedding

print(f"Text: {text_to_embed}")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First five values: {embedding[:5]}")


Text: AI risk management training
Embedding dimensions: 1536
First five values: [-0.0208892822265625, 0.0187530517578125, 0.1077880859375, 0.034576416015625, -0.024810791015625]
